In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, 
    classification_report, 
    ConfusionMatrixDisplay, 
    mean_squared_error, 
    make_scorer, mean_squared_error,  
    mean_absolute_error, r2_score
)
from xgboost import XGBRegressor

In [7]:
eunice = pd.read_csv("../eunice_csv_files/eunice_combined_npml.csv.gz")
nomin = pd.read_csv("../nomin_csv_files/combined_npml_n.csv.gz")
prithvi = pd.read_csv("../prithvi_csv_files/prithvi_combined_npml.csv.gz")
# jade = pd.read_csv("../jade_csv_files/jade_combined_npml.csv.gz")

print(eunice.shape)
print(nomin.shape)
print(prithvi.shape)

(159697, 8)
(159697, 11)
(159697, 9)


In [8]:
eunice.columns
eunice['id'][0]

'3033789_npml_0'

In [9]:
nomin.columns
nomin['id'][2]

'3033791_npml_0'

In [10]:
prithvi.columns
prithvi['id'][0]

np.int64(3033789)

In [11]:
eunice = eunice.drop(columns=["file"], errors="ignore")
nomin = nomin.drop(columns=["file"], errors="ignore")
prithvi = prithvi.drop(columns=["file"], errors="ignore")
# jade = jade.drop(columns=["file"], errors="ignore")

In [12]:
eunice["id"] = eunice["id"].astype(str)
nomin["id"] = nomin["id"].astype(str)
prithvi["id"] = prithvi["id"].astype(str)
# jade["id"] = jade["id"].astype(str)

In [13]:
df = eunice.merge(nomin, on="id")
# df = df.merge(prithvi, on="id")

# Jade merging
# df = df.merge(jade, on="id")

print(df.shape)

(159697, 17)


In [14]:
df

,id,ED,HWP,ND80,PPR,SCA,LQ80,tail_slope_no_pz,current_kurtosis,time_to_peak,total_power,tail_slope,current_skewness,time_to_main_peak,spectral_centroid_power,current_width,tp0
0,3033789_npml_0,3405,2090.0,0.0,0.694995,0.034962,-3.234871e+05,-10061.013124,6.674751,83,8.638530,-4.313729,2.738811,83,104.055619,0.159431,968
1,3033790_npml_0,3406,2165.0,0.0,0.702777,0.035541,-2.097532e+05,-9089.512080,10.563529,180,8.397907,-13.898763,3.335234,180,107.098676,0.121488,942
2,3033791_npml_0,3408,2129.0,0.0,0.699074,0.035238,-2.425070e+05,-9449.901755,5.231941,185,8.444459,2.577768,2.477587,185,103.335168,0.182753,940
3,3033792_npml_0,3411,2125.0,0.0,0.701728,0.035928,-2.223822e+05,-8567.556832,7.421443,201,8.378091,-0.236396,2.862397,201,106.077102,0.154580,936
4,3033793_npml_0,3405,2002.0,0.0,0.684008,0.035634,-2.887550e+05,-10081.558061,8.552893,96,8.453313,3.093208,3.057418,96,109.909804,0.143348,952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159692,3193481_npml_2,3408,2046.0,0.0,0.693882,0.034458,-8.314132e+05,-10371.604799,7.796771,77,9.425819,-1.783430,2.933655,77,108.444858,0.148676,972
159693,3193482_npml_2,3411,2060.0,0.0,0.696157,0.034467,-1.789650e+06,-10700.698164,10.240402,94,10.085858,32.625510,3.286314,94,111.463167,0.122861,967
159694,3193483_npml_2,3409,2082.0,0.0,0.697220,0.034344,-1.315494e+06,-10687.382341,10.704435,86,9.845565,33.048006,3.361456,86,109.771213,0.121086,966
159695,3193484_npml_2,3405,2062.0,0.0,0.693403,0.034225,-2.350121e+06,-10623.611465,8.239963,65,10.337184,0.546147,2.972791,65,106.924402,0.136744,981


## everyone include your classification model here so once our final npml file is done, we can just create the predictions csv in here

### high_avse classification

In [ ]:
high_avse_xgb_threshold = 0.7

high_avse_xgb = xgb.XGBClassifier(
    subsample=1.0,
    scale_pos_weight=0.5,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.3,
    gamma=0.1,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

high_avse_xgb.fit(X_train, y_train)

y_probs_high_avse = high_avse_xgb.predict_proba(X_test)[:, 1]
y_pred_high_avse = (y_probs_high_avse >= high_avse_xgb_threshold).astype(int)

### lq classification

In [ ]:
rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    ))
])

In [ ]:
df_train = pd.read_csv("combined_train_with_labels.csv.gz")

drop_cols = [
    "id",
    "energy_label",
    "psd_label_low_avse",
    "psd_label_high_avse",
    "psd_label_dcr",
    "psd_label_lq"
]

X = df_train.drop(columns=drop_cols)
y = df_train["psd_label_lq"]

rf_pipe.fit(X, y)

In [ ]:
X_npml = df.drop(columns=["id"], errors="ignore")

X_npml = X_npml[X.columns]

scores = rf_pipe.predict_proba(X_npml)[:, 1]
best_threshold = 0.42

pred_lq = (scores >= best_threshold).astype(int)

In [ ]:
pred_df = pd.DataFrame({
    "id": df["id"],
    "predicted_label_lq": pred_lq
})

pred_df.to_csv("npml_lq_predictions.csv", index=False)